# E01 — a promessa e o corte

Este caderno mede uma tentativa que qualquer pessoa faz, e que falha de um jeito
interessante.

**A tentativa.** Olhar o último ano de uma série, pegar os piores dias dela e anunciar:
"amanhã não passo daqui, e isso vale 19 vezes em 20". É a barra de erro mais comum do
mundo.

**O que se mede.** Duas leituras do mesmo corte:

1. a **taxa média** de violações contra a promessa anunciada;
2. o **pior bloco** de 60 dias, que é onde o mundo aparece.

E um controle que separa as duas: um mundo que **nunca muda** (retornos sorteados,
independentes, sempre com a mesma lei), submetido à mesma rotina.

**Convenções** (AGENTS.md §7 e §9): um experimento por caderno, parâmetros no topo marcados
com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E01_promessa.json,
figura em .pdf e .png.

In [1]:
# <- brinque com: SERIE, JANELA, CAUDA, BLOCO, MUNDOS, JANELAS, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, promessa, volatilidade

RAIZ = Path.cwd()
SERIE = "sp500.csv"      # a série do arquivo do projeto anterior (.old/dados/)
JANELA = 252             # o corte: um ano de pregões
CAUDA = 0.05             # a promessa anunciada: 1 dia em 20
BLOCO = 60               # o bloco em que a entrega é conferida
MESES = ("janeiro", "fevereiro", "março", "abril", "maio", "junho", "julho",
         "agosto", "setembro", "outubro", "novembro", "dezembro")
MUNDOS = 200             # quantos mundos que nunca mudam são sorteados
JANELAS = (21, 63, 252, 1260)   # a varredura do corte: um mês, um trimestre, um ano, cinco anos
SEMENTE = 11

retornos = volatilidade.retornos_log(dados.carregar_serie(SERIE))
print("frevolab %s | %s: %d dias, de %s a %s" % (
    frevolab.VERSAO, SERIE, len(retornos),
    retornos.index.min().date(), retornos.index.max().date()))

frevolab 0.1.0 | sp500.csv: 6718 dias, de 2000-01-04 a 2026-09-21


## A promessa e o corte, no mundo real

In [2]:
# A medição no mundo real: as duas leituras, para cada corte da varredura.
medidas = {j: promessa.entrega(retornos, j, CAUDA, BLOCO) for j in JANELAS}

linhas = []
for j in JANELAS:
    d = medidas[j]
    linhas.append({
        "janela": j,
        "corte (k-esimo pior)": promessa.posto(j, CAUDA),
        "dias": d["dias"],
        "violacoes": d["violacoes"],
        "taxa (%)": 100 * d["taxa"],
        "conta do corte (%)": 100 * d["assinado"],
        "mediana do bloco": d["bloco_mediana"],
        "pior bloco": d["bloco_pior"],
        "blocos com o dobro da promessa (%)": 100 * d["bloco_acima_do_dobro"],
    })
tabela = pd.DataFrame(linhas).set_index("janela")
print(tabela.round(3).to_string())
print()
print("promessa anunciada: %.2f%% | prometido em %d dias: %.0f violacoes" % (
    100 * CAUDA, BLOCO, BLOCO * CAUDA))

        corte (k-esimo pior)  dias  violacoes  taxa (%)  conta do corte (%)  mediana do bloco  pior bloco  blocos com o dobro da promessa (%)
janela                                                                                                                                       
21                         2  6697        630     9.407               9.091               6.0          12                              33.173
63                         4  6655        440     6.612               6.250               3.0          15                              15.722
252                       13  6466        332     5.135               5.138               2.0          20                              13.048
1260                      63  5458        278     5.093               4.996               2.0          24                              15.688

promessa anunciada: 5.00% | prometido em 60 dias: 3 violacoes


## O controle: um mundo que nunca muda

In [3]:
# O controle: um mundo que nunca muda, submetido exatamente a mesma rotina.
sorteio = np.random.default_rng(SEMENTE)
taxas_nulas = {j: [] for j in JANELAS}
piores_nulos = {j: [] for j in JANELAS}

for _ in range(MUNDOS):
    mundo = pd.Series(sorteio.normal(0.0, 0.01, len(retornos)), index=retornos.index)
    for j in JANELAS:
        d = promessa.entrega(mundo, j, CAUDA, BLOCO)
        taxas_nulas[j].append(100 * d["taxa"])
        piores_nulos[j].append(d["bloco_pior"])

print("%6s %10s %16s %12s %14s %12s" % (
    "janela", "conta(%)", "nunca muda(%)", "desvio", "pior bloco", "pior real"))
for j in JANELAS:
    print("%6d %10.3f %16.3f %12.3f %14d %12d" % (
        j, 100 * promessa.entrega_do_corte(j, CAUDA), np.mean(taxas_nulas[j]),
        np.std(taxas_nulas[j], ddof=1), max(piores_nulos[j]), medidas[j]["bloco_pior"]))

janela   conta(%)    nunca muda(%)       desvio     pior bloco    pior real
    21      9.091            9.105        0.189             14           12
    63      6.250            6.250        0.135             14           15
   252      5.138            5.139        0.109             13           20
  1260      4.996            4.987        0.127             13           24


## As figuras

In [4]:
# Figura 1: a contagem em blocos ao longo do tempo, com as duas linhas de referencia.
v = promessa.violacoes(retornos, JANELA, CAUDA)
contagem = promessa.conta_em_blocos(v, BLOCO)
pior_nulo = max(piores_nulos[JANELA])

fig, eixo = plt.subplots(figsize=(9.5, 4.3))
eixo.plot(contagem.index, contagem.to_numpy(), lw=0.9, color="#1f4e79")
eixo.axhline(BLOCO * CAUDA, color="#7f7f7f", ls="--", lw=1.2,
             label="o que a promessa de %d%% admite: %.0f em %d dias" % (100 * CAUDA, BLOCO * CAUDA, BLOCO))
eixo.axhline(pior_nulo, color="#b03a2e", ls=":", lw=1.6,
             label="o pior bloco de %d mundos que nunca mudaram: %d" % (MUNDOS, pior_nulo))
eixo.set_ylim(0, max(contagem.max(), pior_nulo) * 1.15)
eixo.set_xlabel("ano")
eixo.set_ylabel("violações nos %d dias anteriores" % BLOCO)
eixo.legend(loc="upper left", frameon=False, fontsize=9)
graficos.salvar(fig, "E01_promessa", 1)
plt.close(fig)
print("pior bloco real: %d (%s) | pior bloco de %d mundos que nunca mudaram: %d" % (
    contagem.max(), contagem.idxmax().date(), MUNDOS, pior_nulo))

pior bloco real: 20 (2008-11-20) | pior bloco de 200 mundos que nunca mudaram: 13


## Leitura visual das figuras

Feita nesta sessão abrindo os .png com a ponte de visão (AGENTS.md §9). O que segue é
observação, e observação não vira número.

**Figura 1.** Linha em degraus, de 2000 a 2026, com o eixo y de 0 a 20. A série passa
trechos longos colada perto de zero e sobe em surtos curtos. O pico mais alto fica por
volta de 2009 e encosta em 20; o segundo, por volta de 2020, chega a 18. A leitura também
aponta surtos intermediários cruzando a linha pontilhada do teto — por volta de 2007, de
2011/2012 e de 2018.

O que o eixo engana: como a linha tracejada da promessa está em 3, e a série vive perto
disso nos trechos calmos, a parte de baixo do gráfico dá a impressão de promessa cumprida.
Só a escala até 20 mostra que os surtos passam longe dela, e um gráfico cortado em 8
esconderia justamente o que importa.

**Figura 2.** Dois painéis com as mesmas quatro janelas no eixo x (21, 63, 252, 1260). À
esquerda, três marcas por janela — a conta do corte, o mundo real e a promessa anunciada
—, com o eixo até 8%; à direita, duas — o pior dos 200 mundos que nunca mudaram e o mundo
real —, com o eixo até 25 violações. A diferença de escala entre os dois painéis é ela
mesma um aviso: o número que o mundo produz não é grande na média, é grande no bloco.

In [5]:
# Figura 2: as duas leituras lado a lado, mundo real contra mundo que nunca muda.
rotulos = [str(j) for j in JANELAS]
x = np.arange(len(JANELAS))
largura = 0.38

fig, (esquerda, direita) = plt.subplots(1, 2, figsize=(9.8, 3.9))

esquerda.bar(x - largura / 2, [100 * promessa.entrega_do_corte(j, CAUDA) for j in JANELAS],
             largura, color="#7f7f7f", label="a conta do corte")
esquerda.bar(x + largura / 2, [100 * medidas[j]["taxa"] for j in JANELAS],
             largura, color="#1f4e79", label="o mundo real")
esquerda.axhline(100 * CAUDA, color="#b03a2e", ls="--", lw=1.3, label="a promessa anunciada")
esquerda.set_xticks(x); esquerda.set_xticklabels(rotulos)
esquerda.set_xlabel("janela do corte (dias)"); esquerda.set_ylabel("violações (%)")
esquerda.set_title("a média: assinada pelo corte", fontsize=10)
esquerda.legend(frameon=False, fontsize=8)

direita.bar(x - largura / 2, [max(piores_nulos[j]) for j in JANELAS],
            largura, color="#7f7f7f", label="pior de %d mundos que nunca mudaram" % MUNDOS)
direita.bar(x + largura / 2, [medidas[j]["bloco_pior"] for j in JANELAS],
            largura, color="#1f4e79", label="o mundo real")
direita.axhline(BLOCO * CAUDA, color="#b03a2e", ls="--", lw=1.3, label="o que a promessa admite")
direita.set_xticks(x); direita.set_xticklabels(rotulos)
direita.set_xlabel("janela do corte (dias)"); direita.set_ylabel("violações no pior bloco")
direita.set_title("o pior bloco: do mundo", fontsize=10)
direita.legend(frameon=False, fontsize=8)

fig.tight_layout()
graficos.salvar(fig, "E01_promessa", 2)
plt.close(fig)
print("figura 2 gravada")

figura 2 gravada


In [6]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
d252 = medidas[JANELA]

# O teto do mundo que nunca mudou: quantas vezes ele e cruzado, e em quantos episodios.
teto = int(max(piores_nulos[JANELA]))
episodios = promessa.episodios_acima(contagem, teto)
acima = contagem > teto
anos = sorted({e["inicio"].year for e in episodios})
texto_anos = ", ".join(str(a) for a in anos[:-1]) + " e " + str(anos[-1])
print("teto de %d violacoes em %d dias: cruzado em %d blocos (%.1f%% de %d) em %d episodios" % (
    teto, BLOCO, int(acima.sum()), 100 * float(acima.mean()), len(contagem), len(episodios)))
for e in episodios:
    print("   %s a %s, pico %d" % (e["inicio"].date(), e["fim"].date(), e["pico"]))
print()

resultado = {
    "promessa_pct": 100 * CAUDA,
    "serie_dias": int(len(retornos)),
    "serie_inicio": "%d de %s de %d" % (retornos.index.min().day, MESES[retornos.index.min().month - 1], retornos.index.min().year),
    "serie_fim": "%d de %s de %d" % (retornos.index.max().day, MESES[retornos.index.max().month - 1], retornos.index.max().year),
    "corte_janela": JANELA,
    "corte_posto": promessa.posto(JANELA, CAUDA),
    "corte_conta_pct": 100 * promessa.entrega_do_corte(JANELA, CAUDA),
    "corte_alternativo_pct": 100 * (promessa.posto(JANELA, CAUDA) - 1) / (JANELA + 1),
    "entrega_dias": d252["dias"],
    "entrega_violacoes": d252["violacoes"],
    "entrega_taxa_pct": 100 * d252["taxa"],
    "bloco_dias": BLOCO,
    "bloco_prometido": int(BLOCO * CAUDA),
    "bloco_mediana": d252["bloco_mediana"],
    "bloco_pior": d252["bloco_pior"],
    "bloco_pior_data": "%d de %s de %d" % (
        pd.Timestamp(d252["bloco_pior_data"]).day,
        MESES[pd.Timestamp(d252["bloco_pior_data"]).month - 1],
        pd.Timestamp(d252["bloco_pior_data"]).year),
    "bloco_acima_do_dobro_pct": 100 * d252["bloco_acima_do_dobro"],
    "nunca_muda_mundos": MUNDOS,
    "nunca_muda_taxa_pct": {"valor": float(np.mean(taxas_nulas[JANELA])),
                            "erro": float(np.std(taxas_nulas[JANELA], ddof=1))},
    "nunca_muda_pior_bloco": int(max(piores_nulos[JANELA])),
    "nunca_muda_pior_medio": float(np.mean(piores_nulos[JANELA])),
    "blocos_acima_do_teto_pct": 100 * float((contagem > max(piores_nulos[JANELA])).mean()),
    "episodios_acima_do_teto": len(episodios),
    "episodios_anos": texto_anos,
    "corte_curto_janela": JANELAS[0],
    "corte_curto_conta_pct": 100 * promessa.entrega_do_corte(JANELAS[0], CAUDA),
    "corte_curto_taxa_pct": 100 * medidas[JANELAS[0]]["taxa"],
    "corte_curto_pior_bloco": medidas[JANELAS[0]]["bloco_pior"],
    "corte_medio_janela": JANELAS[1],
    "corte_medio_conta_pct": 100 * promessa.entrega_do_corte(JANELAS[1], CAUDA),
    "corte_medio_taxa_pct": 100 * medidas[JANELAS[1]]["taxa"],
    "corte_medio_pior_bloco": medidas[JANELAS[1]]["bloco_pior"],
    "corte_longo_janela": JANELAS[3],
    "corte_longo_conta_pct": 100 * promessa.entrega_do_corte(JANELAS[3], CAUDA),
    "corte_longo_taxa_pct": 100 * medidas[JANELAS[3]]["taxa"],
    "corte_longo_pior_bloco": medidas[JANELAS[3]]["bloco_pior"],
}
caminho = Path("lab/resultados/E01_promessa.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True), encoding="utf-8")
print("%s gravado | %d grandezas" % (caminho, len(resultado)))

teto de 13 violacoes em 60 dias: cruzado em 125 blocos (2.0% de 6407) em 6 episodios
   2007-08-15 a 2007-09-05, pico 15
   2007-09-07 a 2007-09-13, pico 14
   2008-10-24 a 2008-12-30, pico 20
   2011-10-03 a 2011-10-03, pico 14
   2018-04-06 a 2018-04-25, pico 15
   2020-03-20 a 2020-05-21, pico 18

lab/resultados/E01_promessa.json gravado | 36 grandezas
